## Databricks Homework
Since in July 2025 Databricks Community Edition was deprecated and instead of creating separate cluster they are being provided in serverless mode it will be easier for you to work with data - since all the data and tables will be saving not only when cluster as active.

So, no separate activities for cluser creating should be executed - it will be autoattached/started when you will execute any of the cells below.


Please, create table in the default schema using file Sales_December_2019.csv. On the left found Catalog => Add Data => Drop files to upload, or click to browse => Sales_December_2019.csv After file will be uploaded, just need to confirm that table should be uploaded.

 Make sure that the first row is header selected => Create Table. Table will be created with name that you specified (sales_december_2019 by default) You will be able to change the table name later if needed.

PySpark can process SQL queries as a text. In other words you don't need to switch cell language to SQL.
1. Write data from table that you created into the dataframe using PySpark with SQL query. Show data in the dataframe

In [0]:
df = spark.sql("SELECT * FROM sales_december_2019")
display(df)


Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
295665,Macbook Pro Laptop,1,1700,12/30/19 00:01,"136 Church St, New York City, NY 10001"
295666,LG Washing Machine,1,600.0,12/29/19 07:03,"562 2nd St, New York City, NY 10001"
295667,USB-C Charging Cable,1,11.95,12/12/19 18:21,"277 Main St, New York City, NY 10001"
295668,27in FHD Monitor,1,149.99,12/22/19 15:13,"410 6th St, San Francisco, CA 94016"
295669,USB-C Charging Cable,1,11.95,12/18/19 12:38,"43 Hill St, Atlanta, GA 30301"
295670,AA Batteries (4-pack),1,3.84,12/31/19 22:58,"200 Jefferson St, New York City, NY 10001"
295671,USB-C Charging Cable,1,11.95,12/16/19 15:10,"928 12th St, Portland, OR 97035"
295672,USB-C Charging Cable,2,11.95,12/13/19 09:29,"813 Hickory St, Dallas, TX 75001"
295673,Bose SoundSport Headphones,1,99.99,12/15/19 23:26,"718 Wilson St, Dallas, TX 75001"
295674,AAA Batteries (4-pack),4,2.99,12/28/19 11:51,"77 7th St, Dallas, TX 75001"


Any notebook can be parameterized using dbutils.widgets. Try to add one parameter "Product_name" and select data from dataframe filtered by value from this parameter. 

2. Select data where product = "product_name" from dataframe using PySpark

In [0]:

dbutils.widgets.text("Product_name", "")

product_name = dbutils.widgets.get("Product_name")


filtered_df = df.filter(df['product'] == product_name)


display(filtered_df)

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address


As well as in SQL, in PySpark you can use aggregate functions. Package pyspark.sql.functions contains all aggregated function from SQL. Try to perform simple aggregation with dataframe. Don't forget, that column types, which you want to calculate, shoud be numerical.  
3. Calculate the sales for each product, including the number of products sold

In [0]:
from pyspark.sql.functions import col, sum

clean_df = df.filter(col("Quantity Ordered") != "Quantity Ordered")

result_df = clean_df.groupBy("Product").agg(
    sum(col("Quantity Ordered").cast("int")).alias("Products Sold"),
    sum(
        col("Quantity Ordered").cast("int") *
        col("Price Each").cast("double")
    ).alias("Total Sales")
)

display(result_df)

Product,Products Sold,Total Sales
Macbook Pro Laptop,644,1094800.0
LG Washing Machine,80,48000.0
USB-C Charging Cable,3251,38849.45000000006
27in FHD Monitor,965,144740.3500000011
AA Batteries (4-pack),3718,14277.120000000394
Bose SoundSport Headphones,1825,182481.74999999825
AAA Batteries (4-pack),4240,12677.599999999413
ThinkPad Laptop,541,540994.5899999965
Lightning Charging Cable,3089,46180.54999999874
Google Phone,716,429600.0


In the PySpark you can perform dataframe profiling using one of two special commands or simple aggregated functions. Try to find special commands to complete this task or just use aggregated functions. Hint: please, сhange the column data types based on the data in them

4. Show data profiles output for the new dataframe of table sales_december_2019_csv: row count, min and max value for each column

In [0]:
from pyspark.sql.functions import col, min, max, count
from pyspark.sql.types import IntegerType, DoubleType


clean_df = df.filter(col("Quantity Ordered") != "Quantity Ordered")


typed_df = clean_df \
    .withColumn("Quantity Ordered", col("Quantity Ordered").cast(IntegerType())) \
    .withColumn("Price Each", col("Price Each").cast(DoubleType()))

print("Row count:", typed_df.count())

profile_df = typed_df.agg(
    min("Order ID").alias("Min Order ID"),
    max("Order ID").alias("Max Order ID"),

    min("Product").alias("Min Product"),
    max("Product").alias("Max Product"),

    min("Quantity Ordered").alias("Min Quantity"),
    max("Quantity Ordered").alias("Max Quantity"),

    min("Price Each").alias("Min Price"),
    max("Price Each").alias("Max Price"),

    min("Order Date").alias("Min Order Date"),
    max("Order Date").alias("Max Order Date"),

    min("Purchase Address").alias("Min Address"),
    max("Purchase Address").alias("Max Address")
)

display(profile_df)

Row count: 24989


Min Order ID,Max Order ID,Min Product,Max Product,Min Quantity,Max Quantity,Min Price,Max Price,Min Order Date,Max Order Date,Min Address,Max Address
295665,319670,20in Monitor,iPhone,1,7,2.99,1700.0,01/01/20 00:10,12/31/19 23:53,"1 12th St, San Francisco, CA 94016","999 West St, Los Angeles, CA 90001"



5. Add new column to the dataframe from previous task with any default value that you want

In [0]:
from pyspark.sql.functions import lit

new_df = typed_df.withColumn("Status", lit("Active"))

display(new_df)

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Status
295665,Macbook Pro Laptop,1,1700.0,12/30/19 00:01,"136 Church St, New York City, NY 10001",Active
295666,LG Washing Machine,1,600.0,12/29/19 07:03,"562 2nd St, New York City, NY 10001",Active
295667,USB-C Charging Cable,1,11.95,12/12/19 18:21,"277 Main St, New York City, NY 10001",Active
295668,27in FHD Monitor,1,149.99,12/22/19 15:13,"410 6th St, San Francisco, CA 94016",Active
295669,USB-C Charging Cable,1,11.95,12/18/19 12:38,"43 Hill St, Atlanta, GA 30301",Active
295670,AA Batteries (4-pack),1,3.84,12/31/19 22:58,"200 Jefferson St, New York City, NY 10001",Active
295671,USB-C Charging Cable,1,11.95,12/16/19 15:10,"928 12th St, Portland, OR 97035",Active
295672,USB-C Charging Cable,2,11.95,12/13/19 09:29,"813 Hickory St, Dallas, TX 75001",Active
295673,Bose SoundSport Headphones,1,99.99,12/15/19 23:26,"718 Wilson St, Dallas, TX 75001",Active
295674,AAA Batteries (4-pack),4,2.99,12/28/19 11:51,"77 7th St, Dallas, TX 75001",Active


Temporary views are processed by cluster and always dropped when the session ends (when the cluster turns off).

6. Create temporary view from task 4 dataframe using PySpark and perform any select using SQL

In [0]:
new_df.createOrReplaceTempView("sales_view")

In [0]:
%sql
SELECT Product, `Quantity Ordered`, Status
FROM sales_view
LIMIT 10;

Product,Quantity Ordered,Status
Macbook Pro Laptop,1,Active
LG Washing Machine,1,Active
USB-C Charging Cable,1,Active
27in FHD Monitor,1,Active
USB-C Charging Cable,1,Active
AA Batteries (4-pack),1,Active
USB-C Charging Cable,1,Active
USB-C Charging Cable,2,Active
Bose SoundSport Headphones,1,Active
AAA Batteries (4-pack),4,Active
